<a href="https://colab.research.google.com/github/DoctorNDJonathan/Regression_Climate/blob/main/01_data_loading_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — Data Loading & Exploratory Data Analysis (EDA)

**Project:** Modelling Global Surface Temperature Change Using Regression  
**Dataset:** NASA GISTEMP v4 — Annual Global Mean Temperature Anomaly (1880–Present)  
**Author:** [Your Name]  
**Date:** 2026-06  
**Environment:** Google Colab (Python 3.10+)

---

### Purpose of this notebook
1. Load the NASA GISTEMP v4 dataset directly from the source URL
2. Inspect the data structure, types, and quality
3. Identify and handle missing values
4. Perform exploratory analysis — trends, distributions, correlations
5. Document key findings that inform model selection in later notebooks

### Dataset citation
> GISTEMP Team, 2026: GISS Surface Temperature Analysis (GISTEMP), version 4.  
> NASA Goddard Institute for Space Studies. https://data.giss.nasa.gov/gistemp/  
> Lenssen et al., 2024: A GISTEMPv4 observational uncertainty ensemble.  
> *J. Geophys. Res. Atmos.*, 129(17), e2023JD040179.


In [ ]:
# =============================================================================
# SECTION 1: IMPORT LIBRARIES
# =============================================================================
# All libraries below are pre-installed in Google Colab — no pip install needed.
# We organise imports by purpose: data, visualisation, statistics.

# --- Data handling ---
import pandas as pd
import numpy as np

# --- Visualisation ---
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# --- Statistics ---
from scipy import stats

# --- Display settings ---
pd.set_option('display.max_columns', 20)      # show all columns in prints
pd.set_option('display.float_format', '{:.4f}'.format)  # 4 decimal places
sns.set_style('whitegrid')                      # clean plot background
plt.rcParams['figure.dpi'] = 120                # sharp figures in Colab

print("All libraries loaded successfully.")


All libraries loaded successfully.


In [ ]:
# =============================================================================
# SECTION 2: CONFIGURATION
# =============================================================================
# Centralise all constants here so they are easy to change in one place.
# This is a best practice — avoids magic numbers scattered through the code.

# URL to the NASA GISTEMP v4 annual global land-ocean temperature index
DATA_URL = "/content/GLB.Ts+dSST (1).csv"

# The GISTEMP baseline period (anomalies are relative to this)
BASELINE_START = 1951
BASELINE_END   = 1980

# Column name we will use as the target variable
TARGET_COL = "J-D"   # 'J-D' = January-to-December annual mean anomaly

# Structural break year for period analysis (Notebook 06)
BREAK_YEAR = 1950

# Random seed for reproducibility across all notebooks
RANDOM_STATE = 42

# Plot colour palette — consistent across all notebooks
COLOR_ACTUAL   = "#3B8BD4"   # blue for observed data
COLOR_TREND    = "#E8593C"   # coral for trend/regression lines
COLOR_BASELINE = "#888780"   # gray for reference lines

print("Configuration set.")
print(f"  Data source : {DATA_URL}")
print(f"  Baseline    : {BASELINE_START}–{BASELINE_END}")
print(f"  Target col  : '{TARGET_COL}' (annual mean anomaly)")


Configuration set.
  Data source : /content/GLB.Ts+dSST (1).csv
  Baseline    : 1951–1980
  Target col  : 'J-D' (annual mean anomaly)


In [ ]:
# =============================================================================
# SECTION 3: DATA LOADING
# =============================================================================
# We wrap data loading in a function so it can be reused across notebooks.
# The NASA CSV has a header row we need to skip, and some cells contain '***'
# for missing data — we handle that during loading.

def load_gistemp(url: str) -> pd.DataFrame:
    """
    Load NASA GISTEMP v4 annual temperature anomaly data from a CSV URL.

    Parameters
    ----------
    url : str
        Direct URL to the GISTEMP CSV file.

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with columns: Year, J-D (annual mean anomaly),
        and individual month columns (Jan–Dec).

    Notes
    -----
    - The CSV uses '***' and '****' to denote missing values.
    - The first row is a header we skip with skiprows=1.
    - All anomaly columns are converted to float (numeric).
    """

    # Step 1: Read CSV, skip the sub-header row, treat '***' as NaN
    df = pd.read_csv(url, skiprows=1, na_values=["***", "****"])

    # Step 2: Keep only the columns we need
    # Columns: Year, Jan, Feb, ..., Dec, J-D, D-N, DJF, MAM, JJA, SON
    # We keep Year + all monthly + annual columns
    month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    annual_cols = ["J-D", "D-N", "DJF", "MAM", "JJA", "SON"]
    keep_cols = ["Year"] + month_cols + annual_cols

    # Only keep columns that actually exist in the CSV
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    # Step 3: Ensure Year is integer (drop any non-numeric trailing rows)
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df.dropna(subset=["Year"])
    df["Year"] = df["Year"].astype(int)

    # Step 4: Convert all anomaly columns to float
    for col in df.columns:
        if col != "Year":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Step 5: Sort by year and reset index
    df = df.sort_values("Year").reset_index(drop=True)

    return df

print("Function load_gistemp() defined.")


Function load_gistemp() defined.


In [ ]:
# =============================================================================
# SECTION 4: LOAD AND PREVIEW THE DATA
# =============================================================================
# Call our loading function and take a first look at the data.

df = load_gistemp(DATA_URL)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Year range   : {df['Year'].min()} to {df['Year'].max()}")
print(f"\nFirst 5 rows:")
df.head()


Dataset loaded: 147 rows × 19 columns
Year range   : 1880 to 2026

First 5 rows:


,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,J-D,D-N,DJF,MAM,JJA,SON
0,1880,-0.1900,-0.2500,-0.1000,-0.1700,-0.1000,-0.2100,-0.1900,-0.1100,-0.1500,-0.2400,-0.2200,-0.1900,-0.1800,NaN,NaN,-0.1200,-0.1700,-0.2000
1,1881,-0.2000,-0.1600,0.0200,0.0400,0.0600,-0.1900,0.0100,-0.0400,-0.1500,-0.2200,-0.1900,-0.0700,-0.0900,-0.1000,-0.1800,0.0400,-0.0700,-0.1800
2,1882,0.1600,0.1400,0.0400,-0.1600,-0.1400,-0.2200,-0.1600,-0.0800,-0.1500,-0.2300,-0.1700,-0.3600,-0.1100,-0.0900,0.0800,-0.0800,-0.1500,-0.1800
3,1883,-0.2900,-0.3600,-0.1200,-0.1800,-0.1700,-0.0700,-0.0700,-0.1400,-0.2200,-0.1100,-0.2400,-0.1100,-0.1700,-0.2000,-0.3400,-0.1600,-0.0900,-0.1900
4,1884,-0.1300,-0.0800,-0.3600,-0.4000,-0.3400,-0.3500,-0.3000,-0.2800,-0.2700,-0.2500,-0.3400,-0.3100,-0.2800,-0.2700,-0.1100,-0.3600,-0.3100,-0.2800


In [ ]:
# =============================================================================
# SECTION 5: DATA INSPECTION FUNCTIONS
# =============================================================================
# Modular functions for inspecting data quality. Reusable in later notebooks.

def inspect_structure(df: pd.DataFrame) -> None:
    """Print the shape, column types, and memory usage of a DataFrame."""

    print("=" * 60)
    print("DATA STRUCTURE")
    print("=" * 60)
    print(f"Rows   : {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(f"Memory : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"\nColumn types:")
    print(df.dtypes.value_counts().to_string())
    print()
    df.info()


def inspect_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a summary of missing values per column.

    Returns a DataFrame with columns: missing_count, missing_pct.
    Only shows columns that have at least one missing value.
    """

    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    summary = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    })
    # Filter to only columns with missing values
    summary = summary[summary["missing_count"] > 0].sort_values(
        "missing_count", ascending=False
    )
    return summary


def inspect_statistics(df: pd.DataFrame, col: str) -> None:
    """Print descriptive statistics for a single numeric column."""

    series = df[col].dropna()
    print(f"\nDescriptive statistics for '{col}':")
    print(f"  Count  : {len(series)}")
    print(f"  Mean   : {series.mean():.4f} °C")
    print(f"  Std    : {series.std():.4f} °C")
    print(f"  Min    : {series.min():.4f} °C  (Year {df.loc[series.idxmin(), 'Year']})")
    print(f"  Max    : {series.max():.4f} °C  (Year {df.loc[series.idxmax(), 'Year']})")
    print(f"  Median : {series.median():.4f} °C")
    print(f"  Skew   : {series.skew():.4f}")
    print(f"  Kurtosis: {series.kurtosis():.4f}")

print("Inspection functions defined: inspect_structure(), inspect_missing(), inspect_statistics()")


Inspection functions defined: inspect_structure(), inspect_missing(), inspect_statistics()


In [ ]:
# =============================================================================
# SECTION 6: RUN DATA INSPECTION
# =============================================================================
# Call the inspection functions on our loaded dataset.

inspect_structure(df)


DATA STRUCTURE
Rows   : 147
Columns: 19
Memory : 21.9 KB

Column types:
float64    18
int64       1

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 19 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Year    147 non-null    int64  
 1   Jan     147 non-null    float64
 2   Feb     147 non-null    float64
 3   Mar     147 non-null    float64
 4   Apr     147 non-null    float64
 5   May     146 non-null    float64
 6   Jun     146 non-null    float64
 7   Jul     146 non-null    float64
 8   Aug     146 non-null    float64
 9   Sep     146 non-null    float64
 10  Oct     146 non-null    float64
 11  Nov     146 non-null    float64
 12  Dec     146 non-null    float64
 13  J-D     146 non-null    float64
 14  D-N     145 non-null    float64
 15  DJF     146 non-null    float64
 16  MAM     146 non-null    float64
 17  JJA     146 non-null    float64
 18  SON     146 non-null    float64
dtypes: float64(1

In [ ]:
# =============================================================================
# SECTION 7: MISSING VALUE ANALYSIS
# =============================================================================
# Check for missing values. The NASA CSV uses '***' for months where data is
# not yet available (e.g., future months in the current year).

missing_summary = inspect_missing(df)

if missing_summary.empty:
    print("No missing values found in any column.")
else:
    print("Columns with missing values:")
    print(missing_summary.to_string())

# Check the target column specifically
target_missing = df[TARGET_COL].isnull().sum()
print(f"\nTarget column '{TARGET_COL}' missing values: {target_missing}")

# If the current year is incomplete, the last row may have NaN in J-D
# We drop rows where the target is missing — these cannot be used for modelling
df_clean = df.dropna(subset=[TARGET_COL]).copy()
print(f"Rows after dropping missing target: {len(df_clean)} (was {len(df)})")


In [ ]:
# =============================================================================
# SECTION 8: DESCRIPTIVE STATISTICS
# =============================================================================
# Compute summary statistics for the annual mean anomaly column.

inspect_statistics(df_clean, TARGET_COL)

# Also show the full describe() output for reference
print("\n" + "=" * 60)
print("FULL STATISTICAL SUMMARY (all numeric columns)")
print("=" * 60)
df_clean.describe().round(4)


In [ ]:
# =============================================================================
# SECTION 9: VISUALISATION — GLOBAL TEMPERATURE TREND
# =============================================================================
# Plot 1: The core time series — annual mean temperature anomaly over time.
# This is the single most important chart in the entire project.

def plot_temperature_trend(df: pd.DataFrame, target: str) -> None:
    """
    Plot the annual mean temperature anomaly as a time series with a
    reference baseline at 0°C (the 1951–1980 average).
    """

    fig, ax = plt.subplots(figsize=(14, 5))

    # Bar chart: colour bars by sign (positive = coral, negative = blue)
    colors = [COLOR_TREND if v >= 0 else COLOR_ACTUAL
              for v in df[target]]
    ax.bar(df["Year"], df[target], color=colors, width=1.0, alpha=0.75,
           edgecolor="none")

    # Baseline reference line at 0°C
    ax.axhline(0, color=COLOR_BASELINE, linewidth=0.8, linestyle="--",
               label=f"Baseline ({BASELINE_START}–{BASELINE_END} avg)")

    # 10-year rolling mean to show the smoothed trend
    rolling = df[target].rolling(window=10, center=True).mean()
    ax.plot(df["Year"], rolling, color="black", linewidth=2,
            label="10-year rolling mean")

    # Labels and formatting
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Temperature anomaly (°C)", fontsize=12)
    ax.set_title("Global mean surface temperature anomaly — 1880 to present",
                 fontsize=14, fontweight="bold")
    ax.legend(loc="upper left", fontsize=10)
    ax.set_xlim(df["Year"].min() - 2, df["Year"].max() + 2)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

# Call the function
plot_temperature_trend(df_clean, TARGET_COL)


In [ ]:
# =============================================================================
# SECTION 10: VISUALISATION — DECADE-BY-DECADE ANALYSIS
# =============================================================================
# Group data by decade to see how average warming has changed over time.
# This directly supports Hypothesis H2 (acceleration of warming).

def analyse_by_decade(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Compute mean, std, min, and max temperature anomaly per decade.

    Returns a summary DataFrame indexed by decade label (e.g. '1880s').
    """

    # Create a decade column: 1880 → 1880, 1991 → 1990, etc.
    df = df.copy()
    df["Decade"] = (df["Year"] // 10) * 10
    df["Decade_label"] = df["Decade"].astype(str) + "s"

    # Group and aggregate
    summary = df.groupby("Decade_label").agg(
        decade_start=("Decade", "first"),
        mean_anomaly=(target, "mean"),
        std_anomaly=(target, "std"),
        min_anomaly=(target, "min"),
        max_anomaly=(target, "max"),
        count=("Year", "count")
    ).sort_values("decade_start")

    return summary


def plot_decade_bars(decade_summary: pd.DataFrame) -> None:
    """Plot a bar chart of mean temperature anomaly per decade."""

    fig, ax = plt.subplots(figsize=(12, 5))

    colors = [COLOR_TREND if v >= 0 else COLOR_ACTUAL
              for v in decade_summary["mean_anomaly"]]

    ax.bar(decade_summary.index, decade_summary["mean_anomaly"],
           color=colors, edgecolor="none", alpha=0.85)

    # Add value labels on bars
    for i, (idx, row) in enumerate(decade_summary.iterrows()):
        ax.text(i, row["mean_anomaly"] + 0.02 * (1 if row["mean_anomaly"] >= 0 else -1),
                f"{row['mean_anomaly']:.2f}",
                ha="center", va="bottom" if row["mean_anomaly"] >= 0 else "top",
                fontsize=9, color=COLOR_BASELINE)

    ax.axhline(0, color=COLOR_BASELINE, linewidth=0.8, linestyle="--")
    ax.set_xlabel("Decade", fontsize=12)
    ax.set_ylabel("Mean anomaly (°C)", fontsize=12)
    ax.set_title("Average temperature anomaly by decade", fontsize=14, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


# Run decade analysis
decade_summary = analyse_by_decade(df_clean, TARGET_COL)
print("Decade summary:")
print(decade_summary[["mean_anomaly", "std_anomaly", "min_anomaly", "max_anomaly", "count"]].round(4).to_string())
print()
plot_decade_bars(decade_summary)


In [ ]:
# =============================================================================
# SECTION 11: CORRELATION ANALYSIS
# =============================================================================
# Compute the Pearson correlation between Year and temperature anomaly.
# A strong positive correlation supports H1 (significant linear trend).

def analyse_correlation(df: pd.DataFrame, x_col: str, y_col: str) -> None:
    """
    Compute Pearson correlation and display a scatter plot with trend line.

    Also performs a statistical significance test (p-value).
    """

    # Pearson correlation with p-value
    r, p_value = stats.pearsonr(df[x_col], df[y_col])

    print(f"Pearson correlation ({x_col} vs {y_col}):")
    print(f"  r       = {r:.4f}")
    print(f"  r²      = {r**2:.4f}  (proportion of variance explained)")
    print(f"  p-value = {p_value:.2e}")
    print(f"  Significant at α=0.05? {'Yes' if p_value < 0.05 else 'No'}")

    # Scatter plot with regression line
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.scatter(df[x_col], df[y_col], s=20, alpha=0.6, color=COLOR_ACTUAL,
               label="Observed annual anomaly")

    # Fit and plot a simple linear trend line using numpy
    slope, intercept = np.polyfit(df[x_col], df[y_col], 1)
    trend_line = slope * df[x_col] + intercept
    ax.plot(df[x_col], trend_line, color=COLOR_TREND, linewidth=2,
            label=f"Linear trend (slope = {slope:.5f} °C/year)")

    ax.set_xlabel(x_col, fontsize=12)
    ax.set_ylabel(f"{y_col} — anomaly (°C)", fontsize=12)
    ax.set_title(f"Scatter: {x_col} vs temperature anomaly (r = {r:.3f})",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print the warming rate per decade
    print(f"\n  Warming rate: {slope * 10:.4f} °C per decade")
    print(f"  Warming rate: {slope * 100:.4f} °C per century")


# Run correlation analysis
analyse_correlation(df_clean, "Year", TARGET_COL)


In [ ]:
# =============================================================================
# SECTION 12: PERIOD COMPARISON — PRE-1950 vs POST-1950
# =============================================================================
# Compare the warming rate before and after 1950 to support RQ3.
# This is a preliminary look — detailed analysis is in Notebook 06.

def compare_periods(df: pd.DataFrame, target: str, break_year: int) -> None:
    """
    Split data at break_year and compare linear slopes for each period.
    """

    pre  = df[df["Year"] <= break_year].copy()
    post = df[df["Year"] > break_year].copy()

    # Fit linear trends for each period
    slope_pre, _ = np.polyfit(pre["Year"], pre[target], 1)
    slope_post, _ = np.polyfit(post["Year"], post[target], 1)

    print(f"Period comparison (break at {break_year}):")
    print(f"  Pre-{break_year}  : {len(pre)} years, slope = {slope_pre:.5f} °C/year ({slope_pre*10:.4f} °C/decade)")
    print(f"  Post-{break_year} : {len(post)} years, slope = {slope_post:.5f} °C/year ({slope_post*10:.4f} °C/decade)")
    print(f"  Ratio        : post-slope is {slope_post/slope_pre:.1f}x the pre-slope")

    # Visualise
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.scatter(pre["Year"], pre[target], s=15, alpha=0.6, color=COLOR_ACTUAL,
               label=f"Pre-{break_year}")
    ax.scatter(post["Year"], post[target], s=15, alpha=0.6, color=COLOR_TREND,
               label=f"Post-{break_year}")

    # Trend lines
    ax.plot(pre["Year"], slope_pre * pre["Year"] + np.polyfit(pre["Year"], pre[target], 1)[1],
            color=COLOR_ACTUAL, linewidth=2, linestyle="--")
    ax.plot(post["Year"], slope_post * post["Year"] + np.polyfit(post["Year"], post[target], 1)[1],
            color=COLOR_TREND, linewidth=2, linestyle="--")

    ax.axvline(break_year, color=COLOR_BASELINE, linewidth=1, linestyle=":",
               label=f"Break year ({break_year})")

    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Anomaly (°C)", fontsize=12)
    ax.set_title(f"Warming rate comparison: pre vs post {break_year}",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


compare_periods(df_clean, TARGET_COL, BREAK_YEAR)


In [ ]:
# =============================================================================
# SECTION 13: DISTRIBUTION ANALYSIS
# =============================================================================
# Check how the temperature anomaly values are distributed.
# Important because linear regression assumes normally distributed residuals.

def plot_distribution(df: pd.DataFrame, target: str) -> None:
    """
    Plot a histogram with KDE overlay and a Q-Q (normal probability) plot.
    """

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Histogram + KDE ---
    axes[0].hist(df[target], bins=25, color=COLOR_ACTUAL, alpha=0.7,
                 edgecolor="white", density=True)
    df[target].plot.kde(ax=axes[0], color=COLOR_TREND, linewidth=2)
    axes[0].set_xlabel("Temperature anomaly (°C)", fontsize=12)
    axes[0].set_ylabel("Density", fontsize=12)
    axes[0].set_title("Distribution of annual anomalies", fontsize=13, fontweight="bold")
    axes[0].grid(alpha=0.3)

    # --- Q-Q Plot ---
    stats.probplot(df[target].dropna(), dist="norm", plot=axes[1])
    axes[1].set_title("Q-Q plot (normality check)", fontsize=13, fontweight="bold")
    axes[1].grid(alpha=0.3)

    # Shapiro-Wilk test for normality
    stat, p_val = stats.shapiro(df[target].dropna())
    axes[1].annotate(f"Shapiro-Wilk p = {p_val:.4f}",
                     xy=(0.05, 0.92), xycoords="axes fraction", fontsize=10,
                     bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))

    plt.tight_layout()
    plt.show()

    # Print interpretation
    print(f"Shapiro-Wilk normality test:")
    print(f"  Statistic = {stat:.4f}")
    print(f"  p-value   = {p_val:.4f}")
    if p_val < 0.05:
        print("  Result: Distribution is NOT normal (p < 0.05)")
        print("  Note: This is expected for trended data. Residual normality")
        print("        (not raw data normality) is what matters for regression.")
    else:
        print("  Result: Distribution is approximately normal (p >= 0.05)")


plot_distribution(df_clean, TARGET_COL)


In [ ]:
# =============================================================================
# SECTION 14: BONUS — MONTHLY ANOMALY HEATMAP
# =============================================================================
# Visualise the monthly anomaly data as a heatmap (years × months).
# This gives a rich visual overview of when warming has been strongest.

def plot_monthly_heatmap(df: pd.DataFrame) -> None:
    """Plot a heatmap of monthly temperature anomalies (year × month)."""

    month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    # Filter to columns that exist
    available = [c for c in month_cols if c in df.columns]
    if not available:
        print("Monthly columns not found — skipping heatmap.")
        return

    # Create the matrix
    heatmap_data = df.set_index("Year")[available]

    # Plot — show every 10th year on the y-axis for readability
    fig, ax = plt.subplots(figsize=(10, 14))
    sns.heatmap(heatmap_data, cmap="RdBu_r", center=0, ax=ax,
                cbar_kws={"label": "Anomaly (°C)", "shrink": 0.6},
                yticklabels=10, linewidths=0, linecolor="white")

    ax.set_title("Monthly temperature anomalies by year",
                 fontsize=14, fontweight="bold")
    ax.set_ylabel("Year", fontsize=12)
    ax.set_xlabel("Month", fontsize=12)

    plt.tight_layout()
    plt.show()


plot_monthly_heatmap(df_clean)


In [ ]:
# =============================================================================
# SECTION 15: SAVE CLEANED DATA FOR LATER NOTEBOOKS
# =============================================================================
# Save the cleaned DataFrame so Notebooks 02–07 can load it directly.
# On Colab, this saves to the session storage (or mount Google Drive).

OUTPUT_PATH = "gistemp_clean.csv"

df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned data saved to: {OUTPUT_PATH}")
print(f"  Rows: {len(df_clean)}, Columns: {len(df_clean.columns)}")
print(f"  Year range: {df_clean['Year'].min()} to {df_clean['Year'].max()}")


---

## Key findings from EDA

**Record these after running all cells above. They inform your model choices in Notebooks 02–07.**

| Finding | Value | Implication |
|---|---|---|
| Dataset size | ___ rows × ___ cols | Small dataset — no need for complex models |
| Missing values in target (J-D) | ___ | Drop incomplete year if any |
| Pearson r (Year vs anomaly) | ___ | If > 0.85, strong linear relationship → H1 supported |
| Warming rate (full period) | ___ °C/decade | Baseline for comparison |
| Pre-1950 slope | ___ °C/decade | Slower warming expected |
| Post-1950 slope | ___ °C/decade | Faster warming expected → H2 supported if ratio > 2x |
| Normality of raw anomalies | ___ (Shapiro-Wilk p) | Non-normal is fine — we test residuals, not raw data |

### Decisions for next notebooks
1. **Notebook 02:** Use Year as sole predictor, J-D as target. Simple linear regression.
2. **Notebook 03:** Engineer features: decade, 10-year rolling mean, lag-1.
3. **Notebook 04:** Polynomial degree 2 and 3 — the decade chart suggests acceleration.
4. **Notebook 05:** Ridge/Lasso after adding engineered features.
5. **Notebook 06:** Separate regressions for pre/post-1950.

---

*End of Notebook 01. Proceed to `02_simple_linear_regression.ipynb`.*
